# SLM-KDN Colab Pro A100: Semantic RAG Micro-KDN

This notebook tests the final architecture:

1. Fine-tuned Llama-3-8B emits strict semantic JSON only.
2. The local command-context/template store retrieves command metadata.
3. Python deterministically assembles Junos CLI.
4. Guardrails enforce commit and mode validity.

Use the older RAG notebook for the baseline `--use_rag` free-form command path. Use this notebook for the final `--semantic_rag` path.

## 1. Check GPU

In [1]:
!nvidia-smi

Tue May 12 00:24:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:


import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Capability: (8, 0)


## 2. Clone Repo From Master

In [3]:
%cd /content
!rm -rf slm-kdn
!git clone --branch master https://github.com/Surya-Prasad/slm-kdn.git
%cd /content/slm-kdn
!git status --short
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

/content
Cloning into 'slm-kdn'...
remote: Enumerating objects: 265, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 265 (delta 155), reused 154 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (265/265), 2.99 MiB | 19.15 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/slm-kdn
master
b1a8f01 (HEAD -> master, origin/master, origin/HEAD) Eval metric


## 3. Install Dependencies

In [4]:
!pip install -q -r requirements.txt
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets pypdf nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.8 MB/s eta 0:00:00


## 4. Hugging Face Login

Run this if your base model or adapter needs gated Hugging Face access.

In [5]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. Preprocess NIT Dataset

This creates `data/processed/train.jsonl`, `val.jsonl`, `test.jsonl`, and robustness variants when available.

In [6]:
!bash scripts/run_preprocess.sh
!find data/processed -maxdepth 1 -type f -print | sort
!head -n 1 data/processed/test.jsonl

README.md: 4.31kB [00:00, 7.87MB/s]
NIT_dataset.json: 273kB [00:00, 185MB/s]
Generating train split: 100% 1000/1000 [00:00<00:00, 120159.97 examples/s]
data/processed/clean_test.jsonl
data/processed/noisy_test.jsonl
data/processed/paraphrased_test.jsonl
data/processed/test_intent_only.jsonl
data/processed/test_intent_with_context.jsonl
data/processed/test.jsonl
data/processed/train_intent_only.jsonl
data/processed/train_intent_with_context.jsonl
data/processed/train.jsonl
data/processed/val_intent_only.jsonl
data/processed/val_intent_with_context.jsonl
data/processed/val.jsonl
{"intent": "How to display the LED status", "context": "show chassis led", "target_command": "show chassis led", "target_json": "{\"action\":\"show\",\"parameters\":{\"prefix\":\"\",\"unit\":0,\"vlan_id\":null,\"vlan_name\":\"\"},\"target\":\"\",\"target_type\":\"unknown\"}", "category": ""}


## 6. Build Command-Context Datastore

This builds `data/juniper_templates.json` from train/val only. It does not use `test.jsonl` unless `--allow-test` is explicitly passed, which you should not use for final evaluation.

In [7]:
!python scripts/build_perfect_datastore_v2.py
!python - <<'PY'
import json
from pathlib import Path
p = Path('data/juniper_templates.json')
payload = json.loads(p.read_text())
print('template_count:', len(payload))
for i, (k, v) in enumerate(payload.items()):
    if i >= 5:
        break
    print(k, '=>', {x: v.get(x) for x in ['template', 'mode', 'requires_commit', 'allowed_params']})
!wc -l outputs/datastore_conflicts.jsonl || true
!head -n 5 outputs/datastore_conflicts.jsonl || true

Wrote 244 templates to /content/slm-kdn/data/juniper_templates.json
Wrote 384 conflicts to /content/slm-kdn/outputs/datastore_conflicts.jsonl
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
template_count: 244
clear/ethernet-switching/port-error/clear_ethernet_switching_port_error/plain => {'template': 'clear ethernet-switching port-error', 'mode': 'operational', 'requires_commit': False, 'allowed_params': []}
set/system/ntp/set_system_ntp/plain => {'template': 'set system ntp server {ip_address} commit comment "configured ntp server"', 'mode': 'configuration', 'requires_commit': True, 'allowed_params': ['description', 'ip_address']}
set/protocols/rstp/set_protocols_rstp/plain => {'template': 'set protocols rstp max-age 15', 'mode': 'configuration', 'requires_commit': True, 'allowed_params': []}
delete/ethernet-switching-options/secure-access-port/delete_ethernet_switching_options_secure_access_port/plain => {'template': 'delete ethernet-switc

## 7. Run Semantic RAG Unit Smoke Tests

In [8]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [9]:
!python tests/test_semantic_rag.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
semantic RAG smoke tests passed


## 8. Train LoRA Adapter

Run this if your current adapter has not been trained in this runtime. For the semantic architecture, the important thing to inspect is the semantic parser metrics below: if JSON validity is low, the adapter is still behaving like the old command-generation model and needs semantic-JSON supervised examples.

In [10]:
!bash scripts/run_train.sh

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
config.json: 100% 654/654 [00:00<00:00, 3.08MB/s]
tokenizer_config.json: 50.6kB [00:00, 95.0MB/s]
tokenizer.json: 9.09MB [00:00, 47.1MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 460kB/s]
model.safetensors.index.json: 23.9kB [00:00, 68.0MB/s]
Fetching 4 files: 100% 4/4 [00:41<00:00, 10.25s/it]
Download complete: 100% 16.1G/16.1G [00:41<00:00, 391MB/s]
Loading weights: 100% 291/291 [00:04<00:00, 64.10it/s]
generation_config.json: 100% 177/177 [00:00<00:00, 928kB/s]
Map: 100% 722/722 [00:00<00:00, 2100.87 examples/s]
{'loss': '0.3473', 'grad_norm': '0.699', 'learning_rate': '0.000187', 'epoch': '0.2216'}
{'loss': '0.1044', 'grad_norm': '0.5625', 'learning_rate': '0.0001725', 'epoch': '0.4432'}
{'loss': '0.06029', 'grad_norm': '1.484', 'learning_rate': '0.000158', 'epoch': '0.6648'}
{'loss': '0.04144', 'grad_norm': '0.4485', 'learning_rate': '0.0001435

## 9. Semantic RAG Smoke Inference

This verifies that `--semantic_rag` writes semantic JSON, command context, assembly errors, commit decisions, and guardrail diagnostics.

In [11]:
import json, os
os.makedirs('data/processed', exist_ok=True)
rows = [
    {'intent': 'Display chassis LED status', 'context': '', 'target_command': 'show chassis led', 'category': 'semantic_smoke'},
    {'intent': 'Clear all MAC address entries in the ethernet switching table', 'context': '', 'target_command': 'clear ethernet-switching-table', 'category': 'semantic_smoke'},
    {'intent': 'Disable the OSPF protocol', 'context': '', 'target_command': 'set protocols ospf disable\\ncommit', 'category': 'semantic_smoke'},
]
with open('data/processed/semantic_rag_smoke.jsonl', 'w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row) + '\n')

!mkdir -p results/predictions results/metrics results/error_analysis
!python src/infer.py \
  --input_file data/processed/semantic_rag_smoke.jsonl \
  --output_file results/predictions/semantic_rag_smoke_predictions.jsonl \
  --semantic_rag \
  --mode intent_with_context

!python scripts/evaluate_semantic_rag.py \
  --pred_file results/predictions/semantic_rag_smoke_predictions.jsonl \
  --out_file results/metrics/semantic_rag_smoke_metrics.json \
  --failures_file results/error_analysis/semantic_rag_smoke_failures.jsonl \
  --summary_file results/error_analysis/semantic_rag_smoke_error_summary.json

!head -n 3 results/predictions/semantic_rag_smoke_predictions.jsonl
!cat results/metrics/semantic_rag_smoke_metrics.json

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:04<00:00, 61.52it/s]

[INFO] Starting BATCHED inference on 3 instances for semantic_rag_smoke.jsonl...
Batches:   0% 0/1 [00:00<?, ?it/s][transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False t

## 10. Full Semantic RAG Evaluation

This runs `test.jsonl` plus `clean_test.jsonl`, `paraphrased_test.jsonl`, and `noisy_test.jsonl` if present. Metrics are written under `results/metrics/`; stage failures and summaries are written under `results/error_analysis/`.

In [12]:
!scripts/run_semantic_rag_eval.sh
!cat results/metrics/semantic_rag_metrics.json
!cat results/error_analysis/semantic_rag_error_summary.json
!head -n 5 results/error_analysis/semantic_rag_failures.jsonl || true

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:04<00:00, 59.62it/s]

[INFO] Starting BATCHED inference on 150 instances for test.jsonl...
Batches:   0% 0/5 [00:00<?, ?it/s][transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress t

## 11. Compare Against Legacy RAG Baseline Optional

Use this only for comparison. The final architecture is `--semantic_rag` above.

In [13]:
!python src/infer.py \
  --input_file data/processed/test.jsonl \
  --output_file results/predictions/rag_predictions_baseline.jsonl \
  --use_rag \
  --rag-corpus train,rag_docs
!python src/evaluate.py \
  --pred_file results/predictions/rag_predictions_baseline.jsonl \
  --out_file results/metrics/rag_baseline_eval_metrics.json
!cat results/metrics/rag_baseline_eval_metrics.json

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[RAG] Building index from:
[RAG]   include data/processed/train.jsonl
[RAG]   include rag-doc/ex3300.pdf
[RAG]   exclude data/processed/val.jsonl
[RAG]   exclude data/processed/test.jsonl
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:05<00:00, 58.12it/s]

[INFO] Starting BATCHED inference on 150 instances for test.jsonl...
Batches:   0% 0/5 [00:00<?, ?it/s][transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_to

## 12. Save Results To Google Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/slm-kdn-results
!cp -r results /content/drive/MyDrive/slm-kdn-results/semantic-rag-results-$(date +%Y%m%d-%H%M%S)
!cp data/juniper_templates.json /content/drive/MyDrive/slm-kdn-results/juniper_templates-$(date +%Y%m%d-%H%M%S).json
!ls -lh /content/drive/MyDrive/slm-kdn-results

Mounted at /content/drive
total 1.1M
-rw------- 1 root root 155K May 11 22:48 juniper_templates-20260511-224857.json
-rw------- 1 root root 155K May 11 23:13 juniper_templates-20260511-231309.json
-rw------- 1 root root 181K May 11 23:37 juniper_templates-20260511-233741.json
-rw------- 1 root root 271K May 12 00:02 juniper_templates-20260512-000206.json
-rw------- 1 root root 273K May 12 00:18 juniper_templates-20260512-001848.json
drwx------ 2 root root 4.0K May 11 18:43 results-rag-20260511-184340
drwx------ 2 root root 4.0K May 11 19:17 results-rag-20260511-191756
drwx------ 2 root root 4.0K May 11 19:43 results-rag-20260511-194342
drwx------ 2 root root 4.0K May 11 20:17 results-rag-20260511-201701
drwx------ 2 root root 4.0K May 11 20:57 results-rag-20260511-205733
drwx------ 2 root root 4.0K May 11 21:11 results-rag-20260511-211105
drwx------ 2 root root 4.0K May 11 22:48 semantic-rag-results-20260511-224855
drwx------ 2 root root 4.0K May 11 23:13 semantic-rag-results-20260511-

In [15]:
!head -n 5 results/predictions/semantic_rag_predictions.jsonl | jq '.semantic_json_raw'

"{\"action\":\"show chassis led\", \"domain\":\"chassis\", \"sub_domain\":\"led\", \"parameters\":{}}\ncommit"
"{\"action\": \"show configuration protocols sflow\", \"domain\": \"configuration\", \"sub_domain\": \"protocols\", \"parameters\": {\"name\": \"sflow\"}}"
"{\"action\":\"set protocols sflow traceoptions flag all\", \"domain\":\"set\", \"sub_domain\":\"protocols sflow traceoptions flag all\", \"parameters\":{}}\ncommit"
"{\"action\":\"set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\", \"domain\": \"set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\", \"sub_domain\": \"set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\", \"parameters\": {}}\ncommit"
"{\"action\":\"set virtual-chassis traceoptions flag csn\", \"domain\":\"set\", \"sub_domain\":\"virtual-chassis traceoptions flag csn\", \"parameters\":{}}\ncommit"


In [16]:
import json, re
from pathlib import Path

pred_file = "results/predictions/semantic_rag_predictions.jsonl"

def norm(x):
    x = str(x or "")
    x = x.replace("\\\\n", "\n")
    x = x.replace("\\n", "\n")
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\s*\n\s*", "\n", x)
    x = x.strip()
    return x

total = 0
raw_em = 0
norm_em = 0
examples = []

for line in Path(pred_file).read_text().splitlines():
    row = json.loads(line)
    p = row.get("prediction", "")
    t = row.get("target_command", "")
    total += 1
    raw_em += int(p == t)
    norm_em += int(norm(p) == norm(t))

    if norm(p) == norm(t) and p != t and len(examples) < 10:
        examples.append({
            "intent": row.get("intent"),
            "prediction_raw": repr(p),
            "target_raw": repr(t),
            "prediction_norm": norm(p),
            "target_norm": norm(t),
        })

print("total", total)
print("raw_em", raw_em, raw_em / total)
print("norm_em", norm_em, norm_em / total)
print("\nexamples where normalized matches but raw fails:")
for e in examples:
    print(json.dumps(e, indent=2))

total 150
raw_em 26 0.17333333333333334
norm_em 61 0.4066666666666667

examples where normalized matches but raw fails:
{
  "intent": "Set sFlow taceoptions to trace all events",
  "prediction_raw": "'set protocols sflow traceoptions flag all\\\\ncommit'",
  "target_raw": "'set protocols sflow traceoptions flag all\\ncommit'",
  "prediction_norm": "set protocols sflow traceoptions flag all\ncommit",
  "target_norm": "set protocols sflow traceoptions flag all\ncommit"
}
{
  "intent": "set a mac moving limit of 2 on vlan HR",
  "prediction_raw": "'set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\\\\ncommit'",
  "target_raw": "'set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\\ncommit'",
  "prediction_norm": "set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\ncommit",
  "target_norm": "set ethernet-switching-options secure-access-port vlan HR mac-move-limit 2\ncommit"
}
{
  "intent": "Trace the virtual chassis co